In [ ]:
#데이터 로드 & 기본 구조 확인
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

print("train:", train.shape)
print("test :", test.shape)

train.head()


In [ ]:
#타깃 분포 확인 (불균형 여부)

target = "completed"

train[target].value_counts(), train[target].value_counts(normalize=True)


In [ ]:
#컬럼 타입 분리 (EDA의 출발점)
dtype_df = (
    train.drop(columns=[target])
    .dtypes
    .value_counts()
    .rename("count")
)

dtype_df


In [ ]:
#범주 / 수치 컬럼 리스트
cat_cols = train.select_dtypes(include=["object", "bool"]).columns.tolist()
num_cols = train.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical:", len(cat_cols))
print("Numerical :", len(num_cols))

cat_cols, num_cols


In [ ]:
missing_ratio = (
    train.isnull()
    .mean()
    .sort_values(ascending=False)
)

missing_ratio[missing_ratio > 0]


In [ ]:
nunique_df = (
    train.drop(columns=[target])
    .nunique(dropna=True)
    .sort_values()
)

nunique_df.head(15)


In [ ]:
prev_cols = [c for c in train.columns if c.startswith("previous_class")]
train[prev_cols].isnull().mean()


In [ ]:
import numpy as np
import pandas as pd

target = "completed"

train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

# -------------------------
# 1) drop 확정 컬럼
# -------------------------
drop_always = ["ID", "contest_award", "idea_contest", "generation", "class4"]

# 조건부: 값은 버리고 "응답했는지"만 사용(추천)
maybe_cols = ["contest_participation", "class3"]

for c in maybe_cols:
    if c in train.columns:
        train[c + "_answered"] = (~train[c].isna()).astype(int)
        test[c + "_answered"]  = (~test[c].isna()).astype(int)

train = train.drop(columns=[c for c in drop_always + maybe_cols if c in train.columns])
test  = test.drop(columns=[c for c in drop_always + maybe_cols if c in test.columns])

# -------------------------
# 2) previous_class 요약 피처 생성
# -------------------------
prev_cols = [c for c in train.columns if c.startswith("previous_class_")]

if len(prev_cols) > 0:
    # 응답(비결측) 개수 = 경험 개수 proxy
    train["previous_class_count"] = train[prev_cols].notna().sum(axis=1)
    test["previous_class_count"]  = test[prev_cols].notna().sum(axis=1)

    train["previous_class_any"] = (train["previous_class_count"] > 0).astype(int)
    test["previous_class_any"]  = (test["previous_class_count"] > 0).astype(int)

    # 원본 컬럼은 일단 제거(노이즈 감소)
    train = train.drop(columns=prev_cols)
    test  = test.drop(columns=prev_cols)

# -------------------------
# 3) 타입 분리 후 결측 처리
# -------------------------
y = train[target].astype(int)
X = train.drop(columns=[target])

cat_cols = X.select_dtypes(include=["object", "bool"]).columns.tolist()
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# 범주형 결측 -> "__MISSING__"
for c in cat_cols:
    X[c] = X[c].astype("object").where(~X[c].isna(), "__MISSING__")
    test[c] = test[c].astype("object").where(~test[c].isna(), "__MISSING__")

# 수치형 결측 -> median
for c in num_cols:
    med = X[c].median()
    X[c] = X[c].fillna(med)
    test[c] = test[c].fillna(med)

print("X shape:", X.shape)
print("test shape:", test.shape)
print("cat:", len(cat_cols), "num:", len(num_cols))
X.head()


In [ ]:
!pip -q install catboost


In [ ]:
#cat_features 인덱스 만들기
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np
import pandas as pd

# cat_cols는 전처리 코드에서 만든 범주형 컬럼 리스트
cat_features = [X.columns.get_loc(c) for c in cat_cols]
len(cat_features), cat_features[:5]


In [ ]:
#불균형 가중치 계산 (F1에 중요)
neg = (y == 0).sum()
pos = (y == 1).sum()
pos_weight = neg / pos
pos_weight

#불균형 가중치 계산 (F1에 중요)
def find_best_threshold(y_true, y_prob, t_min=0.05, t_max=0.95, n=181):
    ts = np.linspace(t_min, t_max, n)
    best_t, best_f1 = 0.5, -1
    for t in ts:
        pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, pred)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1


In [ ]:
#CatBoost 5-Fold 학습 + OOF + Test 평균 예측
RANDOM_STATE = 42
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

oof_prob = np.zeros(len(X))
test_prob = np.zeros(len(test))

params = dict(
    loss_function="Logloss",
    iterations=4000,
    learning_rate=0.03,
    depth=7,
    l2_leaf_reg=5.0,
    random_seed=RANDOM_STATE,
    verbose=200,
    allow_writing_files=False,
    class_weights=[1.0, float(pos_weight)],
)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    tr_pool = Pool(X_tr, y_tr, cat_features=cat_features)
    va_pool = Pool(X_va, y_va, cat_features=cat_features)

    model = CatBoostClassifier(**params)
    model.fit(tr_pool, eval_set=va_pool, early_stopping_rounds=300, use_best_model=True)

    oof_prob[va_idx] = model.predict_proba(va_pool)[:, 1]
    test_prob += model.predict_proba(Pool(test, cat_features=cat_features))[:, 1] / skf.n_splits

    # 참고용: threshold 0.5에서 fold F1
    fold_f1 = f1_score(y_va, (oof_prob[va_idx] >= 0.5).astype(int))
    print(f"[Fold {fold}] F1@0.5 = {fold_f1:.5f}")

best_t, best_f1 = find_best_threshold(y, oof_prob)
print(f"\n[OOF] Best threshold = {best_t:.3f}, Best F1 = {best_f1:.5f}")


In [ ]:
import numpy as np
from sklearn.metrics import f1_score

def find_best_threshold(y_true, y_prob, t_min=0.05, t_max=0.95, n=181):
    ts = np.linspace(t_min, t_max, n)
    best_t, best_f1 = 0.5, -1
    for t in ts:
        pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, pred)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# OneHot: 범주형만 인코딩, 수치형은 그대로 통과
preprocess_ohe = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop",
)

logreg = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",   # 불균형 대응
    solver="liblinear",        # 소형 데이터에서 안정적
)

pipe_lr = Pipeline(steps=[("prep", preprocess_ohe), ("clf", logreg)])

oof_lr = np.zeros(len(X))
test_lr = np.zeros(len(test))

for tr_idx, va_idx in skf.split(X, y):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    pipe_lr.fit(X_tr, y_tr)
    oof_lr[va_idx] = pipe_lr.predict_proba(X_va)[:, 1]
    test_lr += pipe_lr.predict_proba(test)[:, 1] / skf.n_splits

t_lr, f1_lr = find_best_threshold(y, oof_lr)
print(f"[LogReg OOF] best_t={t_lr:.3f}, best_f1={f1_lr:.5f}")


In [ ]:
!pip -q install lightgbm


In [ ]:
import lightgbm as lgb
from sklearn.preprocessing import OrdinalEncoder

# 범주형 -> ordinal (unknown/missing 안전 처리)
ord_enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

X_lgb = X.copy()
T_lgb = test.copy()

X_lgb[cat_cols] = ord_enc.fit_transform(X_lgb[cat_cols])
T_lgb[cat_cols] = ord_enc.transform(T_lgb[cat_cols])

oof_lgb = np.zeros(len(X_lgb))
test_lgb = np.zeros(len(T_lgb))

# 불균형 비율
neg = (y == 0).sum()
pos = (y == 1).sum()
scale_pos_weight = neg / pos

params = dict(
    objective="binary",
    metric="binary_logloss",
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_data_in_leaf=20,
    feature_fraction=0.85,
    bagging_fraction=0.85,
    bagging_freq=1,
    lambda_l2=5.0,
    scale_pos_weight=float(scale_pos_weight),
    verbosity=-1,
    seed=42,
)

for tr_idx, va_idx in skf.split(X_lgb, y):
    X_tr, X_va = X_lgb.iloc[tr_idx], X_lgb.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    dtr = lgb.Dataset(X_tr, label=y_tr)
    dva = lgb.Dataset(X_va, label=y_va)

    model = lgb.train(
        params,
        dtr,
        num_boost_round=5000,
        valid_sets=[dva],
        callbacks=[lgb.early_stopping(300, verbose=False)],
    )

    oof_lgb[va_idx] = model.predict(X_va, num_iteration=model.best_iteration)
    test_lgb += model.predict(T_lgb, num_iteration=model.best_iteration) / skf.n_splits

t_lgb, f1_lgb = find_best_threshold(y, oof_lgb)
print(f"[LGBM OOF] best_t={t_lgb:.3f}, best_f1={f1_lgb:.5f}")


컬럼 정리


In [ ]:
import pandas as pd

train = pd.read_csv("train.csv")
target = "completed"

X = train.drop(columns=[target])

obj_cols = X.select_dtypes(include=["object"]).columns.tolist()

summary = []

for c in obj_cols:
    vc = X[c].dropna().astype(str).value_counts()
    summary.append({
        "column": c,
        "n_nonnull": vc.sum(),
        "n_unique": vc.shape[0],
        "top1": vc.index[0] if vc.shape[0] > 0 else None,
        "top1_cnt": vc.iloc[0] if vc.shape[0] > 0 else 0,
        "top1_ratio": (vc.iloc[0] / vc.sum()) if vc.sum() > 0 else 0
    })

summary_df = pd.DataFrame(summary).sort_values("n_unique")
summary_df


In [ ]:
choice_cols = summary_df[summary_df["n_unique"] <= 10]["column"].tolist()
free_text_cols = summary_df[summary_df["n_unique"] > 10]["column"].tolist()

print("선택형 컬럼 (<=10):", len(choice_cols))
print(choice_cols)

print("\n자유서술형 컬럼 (>10):", len(free_text_cols))
free_text_cols


In [ ]:
for c in choice_cols:
    print(f"\n=== {c} (n_unique <= 10) ===")
    print(X[c].value_counts())


In [ ]:
for c in free_text_cols:
    print(f"\n=== {c} (n_unique > 10) sample ===")
    print(X[c].dropna().astype(str).sample(5, random_state=42).tolist())


In [ ]:
n_unique_df = (
    train
    .nunique(dropna=True)
    .sort_values()
    .reset_index()
)

n_unique_df.columns = ["column", "n_unique"]
n_unique_df


In [ ]:
# =========================
# (A) OOF 기반 Threshold 튜닝
# =========================
from sklearn.metrics import f1_score
import numpy as np

def find_best_threshold(y_true, y_prob, t_min=0.05, t_max=0.95, n=181):
    ts = np.linspace(t_min, t_max, n)
    f1s = []
    for t in ts:
        pred = (y_prob >= t).astype(int)
        f1s.append(f1_score(y_true, pred))
    best_idx = int(np.argmax(f1s))
    return float(ts[best_idx]), float(f1s[best_idx]), ts, np.array(f1s)

# 0.5 기준 F1
f1_05 = f1_score(y, (oof_prob >= 0.5).astype(int))

# 최적 threshold (OOF 전체에 대해)
best_t, best_f1, ts, f1s = find_best_threshold(y, oof_prob, n=201)

print(f"[OOF] F1@0.5 = {f1_05:.4f}")
print(f"[OOF] Best threshold = {best_t:.3f} | Best F1 = {best_f1:.4f}")

# =========================
# (B) 최종 Test 예측 생성 (best_t 적용)
# =========================
test_pred = (test_prob >= best_t).astype(int)
test_pred[:10], test_pred.mean()


In [ ]:
import pandas as pd

# sample_submission 불러오기
sub = pd.read_csv("submission.csv")

# ID 컬럼이 있으면 test에서 가져와서 맞추기 (대부분 이 형태)
if "ID" in sub.columns and "ID" in test.columns:
    sub["ID"] = test["ID"]

# 타겟 컬럼 이름 맞추기: completed
sub["completed"] = test_pred.astype(int)

# 저장
sub.to_csv("submission.csv", index=False, encoding="utf-8")
sub.head()
